# 节点12：LLaMA 与开源爆炸（2023）

本 notebook 用纯 NumPy 手撕 LLaMA 时代最重要的技术：**LoRA（Low-Rank Adaptation）**
以及量化的直觉演示。

> 目标读者：会基础代数 + 基础 Python 的初中生

In [1]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import os

np.random.seed(42)
ASSETS = '../docs/assets'
os.makedirs(ASSETS, exist_ok=True)
print('导入完成')

导入完成


## Part 1：LoRA — 低秩适配的核心思路

想象你有一个超大矩阵 W（模型的权重），形状是 d×d。
全参数微调需要更新 d² 个数；LoRA 只训练两个小矩阵 B（d×r）和 A（r×d），
总参数量只有 2dr 个。

当 r=8，d=1000 时：
- 全参数：1,000,000 个数
- LoRA：16,000 个数（缩小 62.5 倍！）

In [2]:
class LoRALayer:
    """模拟一个 LoRA 适配层（纯 NumPy）。"""

    def __init__(self, d_in: int, d_out: int, rank: int, alpha: float = None):
        self.d_in = d_in
        self.d_out = d_out
        self.rank = rank
        self.alpha = alpha if alpha is not None else 2 * rank
        self.scaling = self.alpha / self.rank

        # 冻结的预训练权重（不会更新）
        self.W0 = np.random.randn(d_out, d_in) * 0.02

        # 可训练的低秩矩阵
        self.A = np.random.randn(rank, d_in) * 0.02   # 高斯初始化
        self.B = np.zeros((d_out, rank))               # 零初始化（保证训练开始时 ΔW=0）

    def forward(self, x: np.ndarray) -> np.ndarray:
        """h = W0 x + scaling * B A x"""
        base_out = self.W0 @ x
        lora_out = self.scaling * (self.B @ (self.A @ x))
        return base_out + lora_out

    def delta_W(self) -> np.ndarray:
        """ΔW = scaling * B @ A"""
        return self.scaling * (self.B @ self.A)

    def num_trainable_params(self) -> int:
        """只有 A 和 B 是可训练的。"""
        return self.A.size + self.B.size

    def num_full_params(self) -> int:
        return self.W0.size


# 示例：d=512，rank=8
layer = LoRALayer(d_in=512, d_out=512, rank=8)
x = np.random.randn(512)
h = layer.forward(x)

print(f'输入维度: {x.shape}')
print(f'输出维度: {h.shape}')
print(f'全参数数量: {layer.num_full_params():,}')
print(f'LoRA 可训练参数: {layer.num_trainable_params():,}')
print(f'压缩比: {layer.num_full_params() / layer.num_trainable_params():.1f}x')

输入维度: (512,)
输出维度: (512,)
全参数数量: 262,144
LoRA 可训练参数: 8,192
压缩比: 32.0x


/var/folders/f7/ly4gmfkx0_j9szgtny55xxd80000gn/T/ipykernel_13207/3129426104.py:20: RuntimeWarning: divide by zero encountered in matmul
  base_out = self.W0 @ x
/var/folders/f7/ly4gmfkx0_j9szgtny55xxd80000gn/T/ipykernel_13207/3129426104.py:20: RuntimeWarning: overflow encountered in matmul
  base_out = self.W0 @ x
/var/folders/f7/ly4gmfkx0_j9szgtny55xxd80000gn/T/ipykernel_13207/3129426104.py:20: RuntimeWarning: invalid value encountered in matmul
  base_out = self.W0 @ x
/var/folders/f7/ly4gmfkx0_j9szgtny55xxd80000gn/T/ipykernel_13207/3129426104.py:21: RuntimeWarning: divide by zero encountered in matmul
  lora_out = self.scaling * (self.B @ (self.A @ x))
/var/folders/f7/ly4gmfkx0_j9szgtny55xxd80000gn/T/ipykernel_13207/3129426104.py:21: RuntimeWarning: overflow encountered in matmul
  lora_out = self.scaling * (self.B @ (self.A @ x))
/var/folders/f7/ly4gmfkx0_j9szgtny55xxd80000gn/T/ipykernel_13207/3129426104.py:21: RuntimeWarning: invalid value encountered in matmul
  lora_out = self.s

## 验证：训练开始时 ΔW = 0

B 初始化为全零，所以 ΔW = scaling × B × A = 0。
这意味着训练开始时，LoRA 层的输出和原始模型完全一致。

In [3]:
delta = layer.delta_W()
print(f'ΔW 的最大绝对值（初始）: {np.abs(delta).max():.6f}')
print(f'期望：接近 0.0（因为 B 全零）')

# 模拟「训练」后 B 获得了非零值
layer.B = np.random.randn(512, 8) * 0.1
delta_after = layer.delta_W()
print(f'"训练"后 ΔW 最大绝对值: {np.abs(delta_after).max():.4f}')
print(f'原始权重 W0 的范数: {np.linalg.norm(layer.W0):.4f}')
print(f'ΔW 的范数: {np.linalg.norm(delta_after):.4f}')
print(f'LoRA 更新幅度 / 原始权重幅度: {np.linalg.norm(delta_after)/np.linalg.norm(layer.W0):.2%}')

ΔW 的最大绝对值（初始）: 0.000000
期望：接近 0.0（因为 B 全零）
"训练"后 ΔW 最大绝对值: 0.0692
原始权重 W0 的范数: 10.2389
ΔW 的范数: 5.6872
LoRA 更新幅度 / 原始权重幅度: 55.54%


/var/folders/f7/ly4gmfkx0_j9szgtny55xxd80000gn/T/ipykernel_13207/3129426104.py:26: RuntimeWarning: divide by zero encountered in matmul
  return self.scaling * (self.B @ self.A)
/var/folders/f7/ly4gmfkx0_j9szgtny55xxd80000gn/T/ipykernel_13207/3129426104.py:26: RuntimeWarning: overflow encountered in matmul
  return self.scaling * (self.B @ self.A)
/var/folders/f7/ly4gmfkx0_j9szgtny55xxd80000gn/T/ipykernel_13207/3129426104.py:26: RuntimeWarning: invalid value encountered in matmul
  return self.scaling * (self.B @ self.A)


## Part 2：Rank 的影响 — 参数量 vs 近似能力

rank r 越大，LoRA 的表达能力越强，但参数量也越多。
实验表明：对于大多数 NLP 任务，r=4 或 r=8 就已经足够。

In [4]:
d = 4096  # 模拟 LLaMA-7B 的隐藏维度
ranks = [1, 2, 4, 8, 16, 32, 64]

full_params = d * d
lora_params = [2 * d * r for r in ranks]
compression = [full_params / lp for lp in lora_params]

print(f'全参数（d={d}）: {full_params:,}')
print()
print(f'{'rank':>6}  {'LoRA参数':>12}  {'压缩比':>10}')
print('-' * 35)
for r, lp, comp in zip(ranks, lora_params, compression):
    print(f'{r:>6}  {lp:>12,}  {comp:>9.1f}x')

全参数（d=4096）: 16,777,216

  rank        LoRA参数         压缩比
-----------------------------------
     1         8,192     2048.0x
     2        16,384     1024.0x
     4        32,768      512.0x
     8        65,536      256.0x
    16       131,072      128.0x
    32       262,144       64.0x
    64       524,288       32.0x


In [5]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# 左图：参数量
ax1.bar([str(r) for r in ranks], lora_params, color='steelblue', alpha=0.8)
ax1.axhline(full_params, color='red', linestyle='--', label=f'全参数 {full_params:,}')
ax1.set_xlabel('LoRA Rank (r)')
ax1.set_ylabel('可训练参数数量')
ax1.set_title('LoRA 参数量 vs Rank')
ax1.legend()
ax1.set_yscale('log')

# 右图：压缩比
ax2.plot([str(r) for r in ranks], compression, 'o-', color='orange', linewidth=2, markersize=8)
ax2.set_xlabel('LoRA Rank (r)')
ax2.set_ylabel('压缩比（全参数 / LoRA参数）')
ax2.set_title('LoRA 压缩比 vs Rank')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{ASSETS}/lora_rank_comparison.png', dpi=150, bbox_inches='tight')
plt.close()
print('图表已保存到 docs/assets/lora_rank_comparison.png')

# 打印 rank=8 的具体数字
idx = ranks.index(8)
print(f'\nrank=8 时: LoRA 参数 = {lora_params[idx]:,}, 压缩比 = {compression[idx]:.0f}x')

/var/folders/f7/ly4gmfkx0_j9szgtny55xxd80000gn/T/ipykernel_13207/3930578598.py:19: UserWarning: Glyph 21487 (\N{CJK UNIFIED IDEOGRAPH-53EF}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/var/folders/f7/ly4gmfkx0_j9szgtny55xxd80000gn/T/ipykernel_13207/3930578598.py:19: UserWarning: Glyph 35757 (\N{CJK UNIFIED IDEOGRAPH-8BAD}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/var/folders/f7/ly4gmfkx0_j9szgtny55xxd80000gn/T/ipykernel_13207/3930578598.py:19: UserWarning: Glyph 32451 (\N{CJK UNIFIED IDEOGRAPH-7EC3}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/var/folders/f7/ly4gmfkx0_j9szgtny55xxd80000gn/T/ipykernel_13207/3930578598.py:19: UserWarning: Glyph 21442 (\N{CJK UNIFIED IDEOGRAPH-53C2}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/var/folders/f7/ly4gmfkx0_j9szgtny55xxd80000gn/T/ipykernel_13207/3930578598.py:19: UserWarning: Glyph 25968 (\N{CJK UNIFIED IDEOGRAPH-6570}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/var/folders/f7/ly4g

图表已保存到 docs/assets/lora_rank_comparison.png

rank=8 时: LoRA 参数 = 65,536, 压缩比 = 256x


## Part 3：低秩近似的误差

给定一个随机矩阵 W，用不同的 rank 对它做低秩近似（SVD分解），
看看 rank 越大，近似越精确。
这验证了 LoRA 背后的线性代数直觉。

In [6]:
def low_rank_approx(W: np.ndarray, rank: int) -> np.ndarray:
    """用 SVD 做低秩近似，等价于 rank-r 截断。"""
    U, s, Vt = np.linalg.svd(W, full_matrices=False)
    # 只保留前 rank 个奇异值
    s_trunc = np.zeros_like(s)
    s_trunc[:rank] = s[:rank]
    return U @ np.diag(s_trunc) @ Vt


# 创建一个「真实权重变化」矩阵（模拟微调后的 ΔW）
d = 128
np.random.seed(42)
# 用低秩结构生成（现实中微调权重确实是低秩的）
true_B = np.random.randn(d, 4)  # 真实秩=4
true_A = np.random.randn(4, d)
W_true = true_B @ true_A

ranks_test = [1, 2, 4, 8, 16]
errors = []
for r in ranks_test:
    W_approx = low_rank_approx(W_true, r)
    err = np.linalg.norm(W_true - W_approx, 'fro') / np.linalg.norm(W_true, 'fro')
    errors.append(err)
    print(f'rank={r:2d}: 相对误差 = {err:.4f}')

print(f'\n（真实矩阵是 rank-4 构造的，所以 rank≥4 时误差为 0）')

rank= 1: 相对误差 = 0.8123
rank= 2: 相对误差 = 0.6362
rank= 4: 相对误差 = 0.0000
rank= 8: 相对误差 = 0.0000
rank=16: 相对误差 = 0.0000

（真实矩阵是 rank-4 构造的，所以 rank≥4 时误差为 0）


/var/folders/f7/ly4gmfkx0_j9szgtny55xxd80000gn/T/ipykernel_13207/4075894074.py:16: RuntimeWarning: divide by zero encountered in matmul
  W_true = true_B @ true_A
/var/folders/f7/ly4gmfkx0_j9szgtny55xxd80000gn/T/ipykernel_13207/4075894074.py:16: RuntimeWarning: overflow encountered in matmul
  W_true = true_B @ true_A
/var/folders/f7/ly4gmfkx0_j9szgtny55xxd80000gn/T/ipykernel_13207/4075894074.py:16: RuntimeWarning: invalid value encountered in matmul
  W_true = true_B @ true_A
/var/folders/f7/ly4gmfkx0_j9szgtny55xxd80000gn/T/ipykernel_13207/4075894074.py:7: RuntimeWarning: divide by zero encountered in matmul
  return U @ np.diag(s_trunc) @ Vt
/var/folders/f7/ly4gmfkx0_j9szgtny55xxd80000gn/T/ipykernel_13207/4075894074.py:7: RuntimeWarning: overflow encountered in matmul
  return U @ np.diag(s_trunc) @ Vt
/var/folders/f7/ly4gmfkx0_j9szgtny55xxd80000gn/T/ipykernel_13207/4075894074.py:7: RuntimeWarning: invalid value encountered in matmul
  return U @ np.diag(s_trunc) @ Vt


In [7]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(ranks_test, errors, 'o-', color='steelblue', linewidth=2, markersize=10)
ax.axvline(x=4, color='red', linestyle='--', alpha=0.7, label='真实 rank=4')
ax.set_xlabel('近似 Rank')
ax.set_ylabel('相对 Frobenius 误差')
ax.set_title('低秩近似误差 vs Rank\n（验证 LoRA 的数学基础）')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xticks(ranks_test)

plt.tight_layout()
plt.savefig(f'{ASSETS}/lora_approx_error.png', dpi=150, bbox_inches='tight')
plt.close()
print('图表已保存')

/var/folders/f7/ly4gmfkx0_j9szgtny55xxd80000gn/T/ipykernel_13207/3212275377.py:11: UserWarning: Glyph 36817 (\N{CJK UNIFIED IDEOGRAPH-8FD1}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/var/folders/f7/ly4gmfkx0_j9szgtny55xxd80000gn/T/ipykernel_13207/3212275377.py:11: UserWarning: Glyph 20284 (\N{CJK UNIFIED IDEOGRAPH-4F3C}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/var/folders/f7/ly4gmfkx0_j9szgtny55xxd80000gn/T/ipykernel_13207/3212275377.py:11: UserWarning: Glyph 30456 (\N{CJK UNIFIED IDEOGRAPH-76F8}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/var/folders/f7/ly4gmfkx0_j9szgtny55xxd80000gn/T/ipykernel_13207/3212275377.py:11: UserWarning: Glyph 23545 (\N{CJK UNIFIED IDEOGRAPH-5BF9}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/var/folders/f7/ly4gmfkx0_j9szgtny55xxd80000gn/T/ipykernel_13207/3212275377.py:11: UserWarning: Glyph 35823 (\N{CJK UNIFIED IDEOGRAPH-8BEF}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/var/folders/f7/ly4g

图表已保存


## Part 4：Self-Instruct 数据格式

Alpaca 用 52K 条 Self-Instruct 格式的数据微调 LLaMA，
成本只有 ~$500。下面演示这种格式。

In [8]:
# Alpaca/Self-Instruct 数据格式示例
alpaca_samples = [
    {
        'instruction': '用一句话解释机器学习和深度学习的区别。',
        'input': '',
        'output': '机器学习是让计算机从数据中学习规律的技术，深度学习是机器学习的子集，使用多层神经网络。'
    },
    {
        'instruction': '将以下英文句子翻译成中文。',
        'input': 'The quick brown fox jumps over the lazy dog.',
        'output': '敏捷的棕色狐狸跳过了懒惰的狗。'
    },
    {
        'instruction': '写一首关于春天的四行诗。',
        'input': '',
        'output': '春风轻抚面，\n樱花漫天飞。\n绿草铺大地，\n万物皆生辉。'
    },
]


def format_alpaca_prompt(sample: dict) -> str:
    """将 Self-Instruct 样本格式化为 Alpaca 风格的训练 prompt。"""
    if sample['input']:
        return (
            f"### 指令:\n{sample['instruction']}\n\n"
            f"### 输入:\n{sample['input']}\n\n"
            f"### 输出:\n{sample['output']}"
        )
    else:
        return (
            f"### 指令:\n{sample['instruction']}\n\n"
            f"### 输出:\n{sample['output']}"
        )


for i, sample in enumerate(alpaca_samples):
    print(f'=== 样本 {i+1} ===')
    print(format_alpaca_prompt(sample))
    print()

=== 样本 1 ===
### 指令:
用一句话解释机器学习和深度学习的区别。

### 输出:
机器学习是让计算机从数据中学习规律的技术，深度学习是机器学习的子集，使用多层神经网络。

=== 样本 2 ===
### 指令:
将以下英文句子翻译成中文。

### 输入:
The quick brown fox jumps over the lazy dog.

### 输出:
敏捷的棕色狐狸跳过了懒惰的狗。

=== 样本 3 ===
### 指令:
写一首关于春天的四行诗。

### 输出:
春风轻抚面，
樱花漫天飞。
绿草铺大地，
万物皆生辉。



## Part 5：量化（Quantization）直觉

QLoRA 用 4-bit NormalFloat 量化原始权重，显存占用从 14GB 降到 3.5GB。
下面演示量化的核心思路：把连续浮点数映射到有限个离散值。

In [9]:
def quantize_to_nbit(weights: np.ndarray, bits: int) -> tuple:
    """简单的线性量化演示（非 NF4，但展示核心思路）。"""
    n_levels = 2 ** bits  # 量化级别数
    w_min, w_max = weights.min(), weights.max()

    # 量化：将浮点数映射到整数
    scale = (w_max - w_min) / (n_levels - 1)
    quantized = np.round((weights - w_min) / scale).astype(int)
    quantized = np.clip(quantized, 0, n_levels - 1)

    # 反量化：从整数恢复浮点数
    dequantized = quantized * scale + w_min

    return quantized, dequantized, scale, w_min


# 生成模拟权重（正态分布，符合神经网络权重的真实分布）
np.random.seed(0)
weights = np.random.randn(1000)  # 1000 个 FP32 权重

print('精度对比：')
print(f'{'精度':>8}  {'原始字节':>10}  {'量化字节':>10}  {'压缩比':>8}  {'平均误差':>10}')
print('-' * 55)

bits_list = [8, 4]
results = {}
for bits in bits_list:
    _, dequant, _, _ = quantize_to_nbit(weights, bits)
    orig_bytes = weights.nbytes
    quant_bytes = orig_bytes // (32 // bits)
    ratio = 32 / bits
    mean_err = np.abs(weights - dequant).mean()
    results[bits] = (dequant, mean_err)
    print(f'{bits:>5}-bit  {orig_bytes:>10}  {quant_bytes:>10}  {ratio:>7.0f}x  {mean_err:>10.5f}')

print(f'\nFP16 对比（无量化误差）: 压缩比 2x')

精度对比：
      精度        原始字节        量化字节       压缩比        平均误差
-------------------------------------------------------
    8-bit        8000        2000        4x     0.00573
    4-bit        8000        1000        8x     0.09585

FP16 对比（无量化误差）: 压缩比 2x


In [10]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

x_range = np.linspace(-3, 3, 1000)
orig_sample = np.random.randn(1000)

for ax, bits in zip(axes, [8, 4]):
    dequant, mean_err = results[bits]
    ax.scatter(weights[:200], dequant[:200], alpha=0.3, s=5, color='steelblue')
    ax.plot([-3, 3], [-3, 3], 'r--', linewidth=1, label='完美恢复')
    ax.set_xlabel('原始权重（FP32）')
    ax.set_ylabel(f'{bits}-bit 反量化值')
    ax.set_title(f'{bits}-bit 量化\n平均误差={mean_err:.5f}')
    ax.legend()
    ax.set_xlim(-3, 3)
    ax.set_ylim(-3, 3)

plt.tight_layout()
plt.savefig(f'{ASSETS}/quantization_comparison.png', dpi=150, bbox_inches='tight')
plt.close()
print('图表已保存到 docs/assets/quantization_comparison.png')

图表已保存到 docs/assets/quantization_comparison.png


/var/folders/f7/ly4gmfkx0_j9szgtny55xxd80000gn/T/ipykernel_13207/1357396522.py:17: UserWarning: Glyph 21407 (\N{CJK UNIFIED IDEOGRAPH-539F}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/var/folders/f7/ly4gmfkx0_j9szgtny55xxd80000gn/T/ipykernel_13207/1357396522.py:17: UserWarning: Glyph 22987 (\N{CJK UNIFIED IDEOGRAPH-59CB}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/var/folders/f7/ly4gmfkx0_j9szgtny55xxd80000gn/T/ipykernel_13207/1357396522.py:17: UserWarning: Glyph 26435 (\N{CJK UNIFIED IDEOGRAPH-6743}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/var/folders/f7/ly4gmfkx0_j9szgtny55xxd80000gn/T/ipykernel_13207/1357396522.py:17: UserWarning: Glyph 37325 (\N{CJK UNIFIED IDEOGRAPH-91CD}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/var/folders/f7/ly4gmfkx0_j9szgtny55xxd80000gn/T/ipykernel_13207/1357396522.py:17: UserWarning: Glyph 65288 (\N{FULLWIDTH LEFT PARENTHESIS}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/var/folders/f7/ly4g

## Part 6：LLaMA vs GPT-3 参数量与性能对比

LLaMA-13B 在大部分 benchmark 上超过了 GPT-3 175B，
证明了「更多数据、更充分训练」比「更大模型」更重要。

In [11]:
import matplotlib.patches as mpatches

models = {
    'GPT-3 175B\n(OpenAI, 2020)': {'params': 175, 'type': 'closed', 'score': 70},
    'LLaMA-7B\n(Meta, 2023)': {'params': 7, 'type': 'open', 'score': 63},
    'LLaMA-13B\n(Meta, 2023)': {'params': 13, 'type': 'open', 'score': 72},
    'LLaMA-33B\n(Meta, 2023)': {'params': 33, 'type': 'open', 'score': 77},
    'LLaMA-65B\n(Meta, 2023)': {'params': 65, 'type': 'open', 'score': 81},
}

names = list(models.keys())
params = [m['params'] for m in models.values()]
scores = [m['score'] for m in models.values()]
colors = ['#ff7f7f' if m['type']=='closed' else '#7fbfff' for m in models.values()]

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.bar(names, scores, color=colors, alpha=0.85, edgecolor='gray')

# 在柱上标注参数量
for bar, param, score in zip(bars, params, scores):
    ax.text(bar.get_x() + bar.get_width()/2., score + 0.5,
            f'{param}B', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_ylabel('综合 Benchmark 分数（示意）')
ax.set_title('LLaMA 系列 vs GPT-3：参数量与性能\n（分数为示意值，基于论文报告的多项 benchmark 平均）')
ax.set_ylim(55, 88)
ax.grid(axis='y', alpha=0.3)

patch_closed = mpatches.Patch(color='#ff7f7f', alpha=0.85, label='闭源模型')
patch_open = mpatches.Patch(color='#7fbfff', alpha=0.85, label='开源模型')
ax.legend(handles=[patch_closed, patch_open])

plt.tight_layout()
plt.savefig(f'{ASSETS}/llama_vs_gpt3.png', dpi=150, bbox_inches='tight')
plt.close()
print('图表已保存')
print('\n关键洞察：LLaMA-13B（130亿参数）在多数任务上超过了 GPT-3（1750亿参数）')
print('代价：充分训练的 1 万亿 tokens（GPT-3 只训练了约 3000 亿 tokens）')

/var/folders/f7/ly4gmfkx0_j9szgtny55xxd80000gn/T/ipykernel_13207/3407558907.py:33: UserWarning: Glyph 32508 (\N{CJK UNIFIED IDEOGRAPH-7EFC}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/var/folders/f7/ly4gmfkx0_j9szgtny55xxd80000gn/T/ipykernel_13207/3407558907.py:33: UserWarning: Glyph 21512 (\N{CJK UNIFIED IDEOGRAPH-5408}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/var/folders/f7/ly4gmfkx0_j9szgtny55xxd80000gn/T/ipykernel_13207/3407558907.py:33: UserWarning: Glyph 20998 (\N{CJK UNIFIED IDEOGRAPH-5206}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/var/folders/f7/ly4gmfkx0_j9szgtny55xxd80000gn/T/ipykernel_13207/3407558907.py:33: UserWarning: Glyph 25968 (\N{CJK UNIFIED IDEOGRAPH-6570}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/var/folders/f7/ly4gmfkx0_j9szgtny55xxd80000gn/T/ipykernel_13207/3407558907.py:33: UserWarning: Glyph 65288 (\N{FULLWIDTH LEFT PARENTHESIS}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/var/folders/f7/ly4g

图表已保存

关键洞察：LLaMA-13B（130亿参数）在多数任务上超过了 GPT-3（1750亿参数）
代价：充分训练的 1 万亿 tokens（GPT-3 只训练了约 3000 亿 tokens）


## Part 7：推理时合并权重（无额外延迟）

LoRA 的一个优雅特性：推理时可以把 ΔW = scaling × B × A 直接加到 W₀ 里，
合并后的权重和普通权重完全一样，**不增加任何推理延迟**。

In [12]:
np.random.seed(42)
d_in, d_out, rank = 256, 256, 8

# 创建 LoRA 层，模拟「训练后」的状态
lora = LoRALayer(d_in=d_in, d_out=d_out, rank=rank, alpha=16)
lora.B = np.random.randn(d_out, rank) * 0.1  # 模拟训练后的 B

# 方式1：LoRA 前向（两次矩阵乘法）
x = np.random.randn(d_in)
out_lora = lora.forward(x)

# 方式2：合并权重后前向（一次矩阵乘法）
W_merged = lora.W0 + lora.delta_W()  # 一次性合并
out_merged = W_merged @ x  # 和普通权重完全一样

print('LoRA 分开计算 vs 合并权重计算：')
print(f'最大差异: {np.abs(out_lora - out_merged).max():.2e}  （应接近机器精度）')
print()
print('结论：推理时合并后，完全等价，无额外计算成本！')
print(f'但如果有多个任务，可以保持 A、B 分开，按需切换（模块化）。')

LoRA 分开计算 vs 合并权重计算：
最大差异: 3.33e-16  （应接近机器精度）

结论：推理时合并后，完全等价，无额外计算成本！
但如果有多个任务，可以保持 A、B 分开，按需切换（模块化）。


/var/folders/f7/ly4gmfkx0_j9szgtny55xxd80000gn/T/ipykernel_13207/3129426104.py:20: RuntimeWarning: divide by zero encountered in matmul
  base_out = self.W0 @ x
/var/folders/f7/ly4gmfkx0_j9szgtny55xxd80000gn/T/ipykernel_13207/3129426104.py:20: RuntimeWarning: overflow encountered in matmul
  base_out = self.W0 @ x
/var/folders/f7/ly4gmfkx0_j9szgtny55xxd80000gn/T/ipykernel_13207/3129426104.py:20: RuntimeWarning: invalid value encountered in matmul
  base_out = self.W0 @ x
/var/folders/f7/ly4gmfkx0_j9szgtny55xxd80000gn/T/ipykernel_13207/3129426104.py:21: RuntimeWarning: divide by zero encountered in matmul
  lora_out = self.scaling * (self.B @ (self.A @ x))
/var/folders/f7/ly4gmfkx0_j9szgtny55xxd80000gn/T/ipykernel_13207/3129426104.py:21: RuntimeWarning: overflow encountered in matmul
  lora_out = self.scaling * (self.B @ (self.A @ x))
/var/folders/f7/ly4gmfkx0_j9szgtny55xxd80000gn/T/ipykernel_13207/3129426104.py:21: RuntimeWarning: invalid value encountered in matmul
  lora_out = self.s

## 本节小结

| 关键概念 | 核心公式 / 数字 |
|----------|----------------|
| LoRA | ΔW = scaling × B × A，训练参数从 d² 降到 2dr |
| QLoRA | 4-bit 量化 + LoRA，65B 模型可在单 48GB GPU 微调 |
| LLaMA-13B | 130亿参数超过 GPT-3 175亿参数，关键：1T tokens 数据 |
| Alpaca | $500 + 52K Self-Instruct 数据 ≈ GPT-3.5 对话能力 |

**开源爆炸的核心逻辑**：
1. LLaMA 权重泄漏 → 任何人都能拿到高质量基础模型
2. LoRA/QLoRA → 消费级 GPU 也能微调
3. Self-Instruct → 廉价生成指令数据
4. 三者叠加 → AI 研究从巨头垄断变为全民参与

**下一步**：[节点13 DPO — 更简洁的对齐（2023）](../docs/13-dpo-2023.md)